- 001 hello world

In [ ]:
from ppp_paths import databulk, dataint


In [ ]:
import pandas as pd
import numpy as np
import gc

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
possiblepairs.dropna(inplace=True)
possiblepairs

In [ ]:
dtypes = {
    'magid': 'str', 
    'patent_id': 'str'
}
pcspairs = pd.read_csv(databulk + '_pcs_oa.csv', dtype=dtypes)
pcspairs = pcspairs[pcspairs['uspto']==1]
pcspairs = pcspairs[pcspairs['confscore']>3]
pcspairs = pcspairs[pcspairs['self']=='isself']
pcspairs['patent'] = pcspairs['patent'].str.removeprefix("us-")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b1")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b2")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-b")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a1")
pcspairs['patent'] = pcspairs['patent'].str.removesuffix("-a2")
pcspairs.rename(columns={'oaid': 'magid'}, inplace=True)
pcspairs.rename(columns={'patent': 'patent_id'}, inplace=True)
pcspairs = pcspairs[['magid', 'patent_id']]
pcspairs['magid'] = pcspairs['magid'].astype(str)
pcspairs.drop_duplicates(inplace=True)
pcspairs.dropna(inplace=True)
pcspairs

In [ ]:
def format_number(x):
    if pd.isna(x):
        return x  # Preserve NaN
    if isinstance(x, float) and x.is_integer():
        return str(int(x))  # Convert to int, then to string
    return str(x)  # Keep as is for non-integer


In [ ]:
possiblepairs_patentsonly = possiblepairs['patent_id'].drop_duplicates().reset_index(drop=True)
possiblepairs_patentsonly
patsandcites = pd.merge(pcspairs, possiblepairs_patentsonly, on='patent_id', how='inner')
del pcspairs
del possiblepairs_patentsonly
gc.collect()
patsandcites.dropna(inplace=True)
patsandcites.drop_duplicates(inplace=True)
patsandcites.rename(columns={'magid': 'patcitedmagid'}, inplace=True)
patsandcites['patcitedmagid'] = patsandcites['patcitedmagid'].astype(str)
patsandcites['numcitesperpat'] = patsandcites.groupby('patent_id')['patcitedmagid'].transform('count')
patsandcites

In [ ]:

possiblepairs_papersonly = possiblepairs['magid'].drop_duplicates().reset_index(drop=True)
possiblepairs_papersonly
papersandcites = pd.read_csv(databulk + "papercitations.zip", sep='\t', compression='zip')
papersandcites = papersandcites.rename(columns={'citingpaperid': 'magid', 'citedpaperid': 'papercitedmagid'})
papersandcites['magid'] = papersandcites['magid'].astype(str)
papersandcites['papercitedmagid'] = papersandcites['papercitedmagid'].astype(str)
papersandcites.dropna(inplace=True)
papersandcites = pd.merge(papersandcites, possiblepairs_papersonly, on='magid', how='inner')
del possiblepairs_papersonly
gc.collect()
# print("Number of unique magid/patent_id combinations:", commoncites[['magid', 'patent_id']].drop_duplicates().shape[0])papersandcites.dropna()
papersandcites = papersandcites.drop_duplicates().reset_index(drop=True)
papersandcites['numcitesperpaper'] = papersandcites.groupby('magid')['papercitedmagid'].transform('count')
papersandcites

In [ ]:
# possiblepairs
# possiblepairs = possiblepairs[['magid', 'patent_id']]
# possiblepairs = possiblepairs.merge(patsandcites, on=['patent_id'], how='left')
# del patsandcites
# gc.collect()
# possiblepairs = possiblepairs.merge(papersandcites, on=['magid'], how='left')
# del papersandcites 
# gc.collect()


In [ ]:
patsandcites.rename(columns={'patcitedmagid': 'citedmagid'}, inplace=True)
papersandcites.rename(columns={'papercitedmagid': 'citedmagid'}, inplace=True)

In [ ]:
commoncites = papersandcites.merge(patsandcites, on=['citedmagid'], how='inner')


In [ ]:
# uniquemagpt = commoncites.copy() 
# uniquemagpt = uniquemagpt[['magid', 'patent_id']]
# uniquemagpt.drop_duplicates
# uniquemagpt.shape


In [ ]:
# possiblepairs['commoncite'] = (possiblepairs['patcitedmagid'] == possiblepairs['papercitedmagid'])
commoncites

In [ ]:
commoncites.drop_duplicates(inplace=True)

In [ ]:
# commoncites = commoncites[['magid', 'patent_id', 'citedmagid', 'numcitesperpaper', 'numcitesperpat']]

grouped_counts = commoncites.groupby(['magid', 'patent_id'])['citedmagid'].nunique().reset_index()
grouped_counts.rename(columns={'citedmagid':'numcommoncites'}, inplace=True)
grouped_counts 


In [ ]:
commoncites = pd.merge(commoncites, grouped_counts, on=['magid', 'patent_id'], how='left')
commoncites

In [ ]:
commoncites['citeoverlapct'] = commoncites['numcommoncites'] / (commoncites['numcitesperpaper'] + commoncites['numcitesperpat'])
commoncites

In [ ]:
commoncites = commoncites[['magid', 'patent_id', 'citeoverlapct']]
commoncites = commoncites.drop_duplicates()
gc.collect()

In [ ]:
pppcommoncites = possiblepairs.merge(commoncites, on=['magid', 'patent_id'], how='left')

In [ ]:
pppcommoncites = pppcommoncites[['magid', 'patent_id', 'citeoverlapct']]
pppcommoncites['citeoverlapct'] = pppcommoncites['citeoverlapct'].fillna(0)
pppcommoncites

In [ ]:
pppcommoncites.value_counts('citeoverlapct')

In [ ]:
pppcommoncites.to_parquet(dataint + "possiblepairs_citationoverlap.parquet", compression='snappy')